# Network Packet Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install scapy

from scapy.all import rdpcap, IP, TCP, UDP
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
packets = rdpcap(filename)

data = []
for pkt in packets:
    if IP in pkt:
        proto = None
        sport, dport = None, None
        if TCP in pkt:
            proto = 'TCP'
            sport = pkt[TCP].sport
            dport = pkt[TCP].dport
        elif UDP in pkt:
            proto = 'UDP'
            sport = pkt[UDP].sport
            dport = pkt[UDP].dport
        data.append({
            'src_ip': pkt[IP].src,
            'dst_ip': pkt[IP].dst,
            'proto': proto,
            'sport': sport,
            'dport': dport,
            'length': len(pkt)
        })

df = pd.DataFrame(data)
print(df['src_ip'].value_counts().head(10))
print(df['dport'].value_counts().head(10))
df['proto'].value_counts().plot(kind='bar', title='Protocol Usage')
plt.show()

suspect_ips = df['src_ip'].value_counts()
suspect_ports = df['dport'].value_counts()
potential_scanners = suspect_ips[suspect_ips > 100]
print(df[df['src_ip'].isin(potential_scanners.index)].groupby('src_ip')['dport'].nunique().sort_values(ascending=False))
